**📓 Open this notebook in your IDE**

To properly run this notebook, make sure you've opened your IDE (VS Code or Cursor) from the `genai-agentcore-demos/` directory:

```bash
cd genai-agentcore-demos
cursor .  # Or: code .
```

Then navigate to this notebook in your IDE's file explorer: `finance-personal-assistant/workshop/lab1-develop_a_personal_budget_assistant_strands_agent.ipynb`

**Why?** Your IDE needs to find the virtual environment (`.venv`) at the project root to detect the correct Python kernel.

---

# Lab 1: Develop a personal budget assistant Strands agent

## Overview

In this lab, you'll create a sophisticated personal budget assistant using Strands Agents. We'll start with a basic conversational agent and progressively enhance it with advanced capabilities including model configuration, conversation management, custom tools, and structured outputs.

This lab demonstrates core Strands Agents concepts through practical implementation, showing how each feature builds upon the previous one to create a comprehensive financial advisory system. By the end, you'll have a production-ready agent capable of providing personalized budgeting advice, spending analysis, and financial recommendations.

We're creating a comprehensive **Budget Agent** that helps users manage their personal finances through intelligent conversation and specialized tools. The agent will provide budgeting guidance, analyze spending patterns, and offer actionable financial advice.

![architecture](./images/single-agent.png)

### Budget Agent Tools & Capabilities

| Tool | Description | Example Use Case |
|------|-------------|------------------|
| **calculate_budget** | Calculates 50/30/20 budget breakdown based on monthly income | "I make $5000/month, create a budget for me" |
| **create_financial_chart** | Generates pie charts and visualizations of financial data | "Visualize my spending across different categories" |
| **calculator** | Performs mathematical calculations for financial planning | "Calculate 15% of my monthly income for savings" |

### Agent Features Summary

Our Budget Agent will include:

- **Personalized Financial Guidance**: Tailored advice based on income and spending patterns
- **Interactive Budgeting**: Real-time budget calculations using the proven 50/30/20 rule
- **Visual Analytics**: Chart generation for better financial data comprehension
- **Conversation Memory**: Context retention across multiple interactions for personalized experiences
- **Structured Reporting**: Consistent, parseable financial reports with health scores and recommendations
- **Responsible AI**: Built-in guardrails and disclaimers for ethical financial advice

The agent focuses exclusively on budgeting and spending analysis, providing practical, actionable guidance without investment advice. It serves as the foundation for more complex multi-agent systems you'll build in subsequent labs.

In [1]:
# Install required dependencies for Strands agents and tools
# If you already have the environment created, you can skip this step
!uv sync --upgrade --dev

Resolved 130 packages in 1.11s                                       
⠙ Preparing packages... (0/21)                                                  
⠙ Preparing packages... (0/21)-------------     0 B/417.50 KiB          
⠙ Preparing packages... (0/21)----------------------     0 B/417.50 KiB 
aws-cdk-cloud-assembly-schema ------------------------------     0 B/202.80 KiB
⠙ Preparing packages... (0/21)----------------------     0 B/417.50 KiB 
aws-cdk-cloud-assembly-schema ------------------------------     0 B/202.80 KiB
⠙ Preparing packages... (0/21)----------------------     0 B/417.50 KiB 
aws-cdk-cloud-assembly-schema ------------------------------     0 B/202.80 KiB
⠙ Preparing packages... (0/21)----------------------     0 B/417.50 KiB 
frozendict                    ------------------------------     0 B/15.88 KiB
aws-cdk-cloud-assembly-schema ------------------------------     0 B/202.80 KiB
⠙ Preparing packages... (0/21)----------------------     0 B/417.50 KiB 
frozendict  

In [1]:
# Import core Strands components and utilities for budget agent
import time

import matplotlib.pyplot as plt
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

from utils import create_guardrail, pretty_print_messages

### Associate Amazon Bedrock Guardrail with Strands

Amazon Bedrock provides a [built-in guardrails framework](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html) that integrates directly with the Strands Agents SDK. If a guardrail is triggered, the Strands Agents SDK will automatically overwrite the user's input in the conversation history. This is done so that follow-up questions are not also blocked by the same questions. This can be configured with the guardrail_redact_input boolean, and the guardrail_redact_input_message string to change the overwrite message. Additionally, the same functionality is built for the model's output, but this is disabled by default. You can enable this with the guardrail_redact_output boolean, and change the overwrite message with the guardrail_redact_output_message string. 

In [2]:
# Create Bedrock guardrail for content filtering and safety
guardrail_id, guardrail_arn = create_guardrail()

Using existing guardrail: guardrail-no-gambling-advice


![guardrail](./images/guardrail.png)

Below is an example of how to leverage Bedrock guardrails in your code:

In [3]:
# Configure Bedrock model with Claude 4.5 Sonnet and guardrails
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name="us-west-2",
    temperature=0.0,  # Deterministic responses for financial advice
    guardrail_id=guardrail_id,  # Your Bedrock guardrail ID
    guardrail_version="DRAFT",  # Guardrail version
    guardrail_trace="enabled",
)

In [4]:
# Create basic agent with configured Bedrock model
agent = Agent(model=bedrock_model)

In [5]:
# Test basic agent functionality with general question
response_1 = agent("Hello! What can you do?")

Hello! I'm a language model designed to assist with a variety of tasks such as:

1. **Answering Questions**: Providing information on a wide range of topics.
2. **Writing Assistance**: Helping with essays, reports, and creative writing.
3. **Language Translation**: Translating text between different languages.
4. **Summarization**: Condensing long articles or documents into shorter summaries.
5. **Coding Help**: Offering assistance with programming-related queries.
6. **Recommendations**: Suggesting books, movies, and other media.
7. **General Conversation**: Engaging in casual or topic-specific discussions.

Feel free to ask me anything!

In [6]:
# Test guardrail blocking investment advice (should be filtered)
response_2 = agent("Gambling investment advice")

I apologize, but I'm not able to provide advice or information about that topic. As a financial advisor, I can help with budgeting, investing, savings, and other responsible financial planning topics. How can I assist you with your financial goals?

In [7]:
# Display conversation history with formatting
pretty_print_messages(messages=agent.messages)


💬 CONVERSATION HISTORY (4 messages)

👤 MESSAGE 1 (USER):
----------------------------------------
  Hello! What can you do?

🤖 MESSAGE 2 (ASSISTANT):
----------------------------------------
  Hello! I'm a language model designed to assist with a variety of tasks such as:
  
  1. **Answering Questions**: Providing information on a wide range of topics.
  2. **Writing Assistance**: Helping with essays, reports, and creative writing.
  3. **Language Translation**: Translating text between different languages.
  4. **Summarization**: Condensing long articles or documents into shorter summaries.
  5. **Coding Help**: Offering assistance with programming-related queries.
  6. **Recommendations**: Sugges
    ... [content truncated]

👤 MESSAGE 3 (USER):
----------------------------------------
  [User input redacted.]

🤖 MESSAGE 4 (ASSISTANT):
----------------------------------------
  I apologize, but I'm not able to provide advice or information about that topic. As a financial advisor, I 

## Create Budget Agent

### Step 1: Define a System Prompt

In [10]:
# Define system prompt for budget-focused financial assistant
BUDGET_SYSTEM_PROMPT = """You are a helpful personal finance assistant. 
You provide general strategies for creating budgets, tips on financial discipline to achieve financial milestones, and analyze financial trends. 
You do not provide any investment advice. Keep responses concise and actionable. Always provide 2-3 specific steps the user can take. Focus on practical budgeting and spending advice.
"""

In [11]:
# Create budget agent with custom system prompt
budget_agent_sys = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,  # Associate a system prompt
)

In [12]:
# Test new finance agent functionality with general question
budget_agent_sys("Hello! What can you do?")

Hello! I'm your personal finance assistant. I can help you with:

**Budgeting & Planning**
- Create customized budgets based on your income and expenses
- Develop strategies to save for specific financial goals
- Suggest ways to reduce spending in different categories

**Financial Discipline**
- Provide tips to stick to your budget and build better money habits
- Help you prioritize financial goals (emergency fund, debt payoff, savings)
- Offer accountability strategies and tracking methods

**Financial Analysis**
- Review your spending patterns and identify areas for improvement
- Analyze trends in your finances over time
- Help you understand where your money is going

**What I Don't Do**
- I don't provide investment advice or recommend specific stocks, funds, or investment products

How can I help you with your finances today? Are you looking to create a budget, work toward a specific goal, or better understand your spending?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Hello! I'm your personal finance assistant. I can help you with:\n\n**Budgeting & Planning**\n- Create customized budgets based on your income and expenses\n- Develop strategies to save for specific financial goals\n- Suggest ways to reduce spending in different categories\n\n**Financial Discipline**\n- Provide tips to stick to your budget and build better money habits\n- Help you prioritize financial goals (emergency fund, debt payoff, savings)\n- Offer accountability strategies and tracking methods\n\n**Financial Analysis**\n- Review your spending patterns and identify areas for improvement\n- Analyze trends in your finances over time\n- Help you understand where your money is going\n\n**What I Don't Do**\n- I don't provide investment advice or recommend specific stocks, funds, or investment products\n\nHow can I help you with your finances today? Are you looking to create a budget, work toward a 

In [13]:
# Test budget agent with dining expense analysis
response_3 = budget_agent_sys(
    "I spend $800/month on dining out. Is this too much for someone making $5000/month?"
)

Yes, **$800/month (16% of your income) is quite high** for dining out. Financial experts typically recommend spending **5-10% of income on dining out**, which would be $250-500 in your case.

**Here's the impact:**
- You're spending nearly **$10,000/year** on restaurants
- That extra $300-550/month could build a **$3,600-6,600 emergency fund in one year**
- Or pay off significant debt/boost savings

**3 Steps to Reduce This:**

1. **Set a realistic target**: Start by cutting to $500/month (still allows flexibility), then work toward $400. That saves you $300-400/month immediately.

2. **Implement the "cook 2x, dine 1x" rule**: For every restaurant meal, commit to cooking two meals at home. Meal prep on Sundays for easy weeknight dinners.

3. **Track and categorize**: Use an app or spreadsheet to log every dining expense. Separate "convenience" meals (grabbing lunch at work) from "social" meals (dinner with friends). Cut the convenience ones first—they're usually less satisfying anyway.

### Step 2: Add Conversation Manager

In the Strands Agents SDK, context refers to the information provided to the agent for understanding and reasoning. This includes:

- User messages
- Agent responses
- Tool usage and results
- System prompts

As conversations grow, managing this context becomes increasingly important for several reasons:

- **Token Limits**: Language models have fixed context windows (maximum tokens they can process)
- **Performance**: Larger contexts require more processing time and resources
- **Relevance**: Older messages may become less relevant to the current conversation
- **Coherence**: Maintaining logical flow and preserving important information

#### Conversation Manager Types

Strands Agents provides three types of conversation managers to handle different context management needs:

1. **[SlidingWindowConversationManager](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.conversation_manager.sliding_window_conversation_manager.SlidingWindowConversationManager)** (Default): Implements a sliding window strategy that maintains a fixed number of recent message pairs, automatically removing the oldest when the limit is reached. This is the default conversation manager used by the Agent class and is ideal for most applications where recent context is most important.

2. **[NullConversationManager](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.conversation_manager.null_conversation_manager.NullConversationManager)**: A simple implementation that does not modify the conversation history. It's useful for short conversations that won't exceed context limits, debugging purposes, or cases where you want to manage context manually.

3. **[SummarizingConversationManager](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.conversation_manager.summarizing_conversation_manager.SummarizingConversationManager)**: Implements intelligent conversation context management by summarizing older messages instead of simply discarding them. This approach preserves important information while staying within context limits, making it ideal for long-running conversations where historical context matters.

In [14]:
# Import conversation manager for context handling
from strands.agent.conversation_manager import SummarizingConversationManager

In [15]:
# Configure conversation manager to handle long conversations
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.5,  # Summarize 50% of messages when context reduction is needed
    preserve_recent_messages=10,  # Always keep 10 most recent messages
)

In [16]:
# Create agent with conversation management capabilities
budget_agent_manager = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,
    conversation_manager=conversation_manager,  # Associate a conversation manager
)

In [17]:
# Test budget agent with dining expense analysis
response_3 = budget_agent_manager(
    "Hi"
)

Hello! I'm here to help you with budgeting, saving strategies, and building better financial habits. 

Whether you want to:
- Create or improve your budget
- Find ways to cut expenses
- Build an emergency fund
- Work toward a financial goal
- Develop better spending habits

Just let me know what's on your mind, and I'll provide practical steps to help you move forward!

In [18]:
# Test budget agent with dining expense analysis
response_3 = budget_agent_manager(
    "what was my previous message?"
)

Your previous message was "Hi" - that was your first message to me in this conversation.

Is there a specific financial topic or budgeting question I can help you with today?

### Step 3: Async Iterators for Streaming

Strands Agents SDK provides support for asynchronous iterators through the [stream_async](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.agent.Agent.stream_async) method, enabling real-time streaming of agent responses in asynchronous environments like web servers, APIs, and other async applications.

In [20]:
# Demonstrate streaming responses with async iterator
async for event in budget_agent_manager.stream_async(
    "I make $5000/month and spend $800 on dining out. Is this too much?"
):
    if "data" in event:
        # Only stream text chunks to the client
        print(event["data"], end="")
        time.sleep(0.09) # show streaming properly #NOTE: Remove on production

YesYes, that's quite high. You're spending **16% of your income** on dining out,, that's quite high. You're spending **16% of your income** on dining out, when financial experts typically recommend **5-10% for all food** (groc when financial experts typically recommend **5-10% for all food** (groceries + dining).

**Here are 2-3 steps to improve this:**

1. **Set aeries + dining).

**Here are 2-3 steps to improve this:**

1. **Set a realistic target** - Aim to reduce dining out to $400-500/month (8-10%). This saves realistic target** - Aim to reduce dining out to $400-500/month (8-10%). This saves you $300-400 monthly ($3,600-4,800/ you $300-400 monthly ($3,600-4,800/year).

2. **Track and limit frequency** - Count how many timesyear).

2. **Track and limit frequency** - Count how many times you eat out weekly. If it's daily, cut to 3-4 times per week. Use a bu you eat out weekly. If it's daily, cut to 3-4 times per week. Use a budgeting app or simple spreadsheet to monitor.

3. **Meal

### Step 4: Add Financial Tools 

In [21]:
# Define custom tool for 50/30/20 budget calculations
@tool
def calculate_budget(monthly_income: float) -> str:
    """Calculate 50/30/20 budget breakdown for the given monthly income."""
    needs = monthly_income * 0.50
    wants = monthly_income * 0.30
    savings = monthly_income * 0.20
    return f"💰 Budget for ${monthly_income:,.0f}/month:\n• Needs: ${needs:,.0f} (50%)\n• Wants: ${wants:,.0f} (30%)\n• Savings: ${savings:,.0f} (20%)"

In [23]:
# Define tool for creating financial pie charts
@tool
def create_financial_chart(
    data_dict: dict, chart_title: str = "Financial Chart"
) -> str:
    """Create a pie chart visualization from financial data dictionary."""
    if not data_dict:
        return "❌ No data provided for chart"

    labels = list(data_dict.keys())
    values = list(data_dict.values())
    colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", "#FF9FF3"]

    plt.figure(figsize=(8, 6))
    plt.pie(
        values,
        labels=labels,
        autopct="%1.1f%%",
        colors=colors[: len(values)],
        startangle=90,
    )
    plt.title(f"📊 {chart_title}", fontsize=14, fontweight="bold")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()

    return f"✅ {chart_title} visualization created!"

In [24]:
# Create complete budget agent with all tools integrated
budget_agent = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,
    conversation_manager=conversation_manager,
    tools=[calculate_budget, create_financial_chart, calculator],
)

In [25]:
# Test tool-enabled agent with streaming response
async for event in budget_agent.stream_async(
    "I make $5000/month and spend $800 on dining out. Is this too much?"
):
    if "data" in event:
        # Only stream text chunks to the client
        print(event["data"], end="")
        time.sleep(0.03) # show streaming properly #NOTE: Remove on production


Tool #1: calculate_budget

Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ (800/5000)*100      │                                                                            │
│  │ Result    │ 16                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

****Yes, $800/month on dining out is too highYes, $800/month on dining out is too high.** You're spending **16% of your gross income** on restaurants.** You're spending **16% of your gross income** on restaurants alone, which is eating into your entire "wants alone, which is eating into your entire "wants" budget.

Using the 50/30/20 rule" budget.

Using the 50/30/20 rule for your $5,000 income:
- **Needs**: $2,500  for your $5,000 income:
- **Needs**: $2,500 (housing, utilities, groceries, transportation)
- **(housing, utilities, groceries, transportation)
- **Wants**: $1,500 (dining out, entertainment, hobbies)Wants**: $1,500 (dining out, entertainment, hobbies)
- **Savings**: $1
- **Savings**: $1,000

Your dining spending alone takes,000

Your dining spending alone takes up more than half (53%) of your entire " up more than half (53%) of your entire "wants" category, leaving little room for other discretwants" category, leaving little room for other discretionary spending.

**Here a

### Step 5: Add Structured Output for Financial Reports

In [27]:
# Import Pydantic for structured output models
from typing import List

from pydantic import BaseModel, Field

In [28]:
# Define Pydantic models for structured financial reports
class BudgetCategory(BaseModel):
    name: str = Field(description="Budget category name")
    amount: float = Field(description="Dollar amount for this category")
    percentage: float = Field(description="Percentage of total income")


class FinancialReport(BaseModel):
    monthly_income: float = Field(description="Total monthly income")
    budget_categories: List[BudgetCategory] = Field(
        description="List of budget categories"
    )
    recommendations: List[str] = Field(description="List of specific recommendations")
    financial_health_score: int = Field(
        ge=1, le=10, description="Financial health score from 1-10"
    )

In [29]:
# Generate structured financial report using Pydantic model
structured_response = budget_agent(
    prompt="Generate a comprehensive financial report for someone earning $6000/month with $800 dining expenses.",
    structured_output_model=FinancialReport
).message.get("content")[0].get("toolUse").get("input")


Tool #3: calculate_budget

Tool #4: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ (800/6000)*100      │                                                                            │
│  │ Result    │ 10                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Tool #5: FinancialReport


In [31]:
structured_response

{'monthly_income': 6000,
 'budget_categories': [{'name': 'Needs (Housing, Utilities, Groceries, Transportation)',
   'amount': 3000,
   'percentage': 50},
  {'name': 'Wants (Entertainment, Hobbies, Shopping)',
   'amount': 1000,
   'percentage': 16.7},
  {'name': 'Dining Out', 'amount': 800, 'percentage': 13.3},
  {'name': 'Savings & Debt Repayment', 'amount': 1200, 'percentage': 20}],
 'recommendations': ['Reduce dining out from $800 to $500/month (8.3% of income) to free up $300 for emergency fund or retirement savings',
  "Implement the 'cook 5, dine 2' rule: cook at home 5 days per week and allow dining out only 2 days to control restaurant spending",
  'Set up automatic transfers of $1,200/month to savings on payday - aim for 3-6 months emergency fund ($18,000-$36,000) as first priority',
  'Track all discretionary spending for 60 days using a budgeting app to identify additional areas where you can optimize spending',
  'Consider meal prepping on weekends to reduce weeknight dini

In [30]:
# Display structured report output in formatted way
print(f"Income: ${structured_response['monthly_income']:,.0f}")
for category in structured_response['budget_categories']:
    print(f"• {category['name']}: ${category['amount']:,.0f} ({category['percentage']:.1f}%)")
print(f"\nFinancial Health Score: {structured_response['financial_health_score']}/10")
print("\nRecommendations:")
for i, rec in enumerate(structured_response['recommendations'], 1):
    print(f"{i}. {rec}")

Income: $6,000
• Needs (Housing, Utilities, Groceries, Transportation): $3,000 (50.0%)
• Wants (Entertainment, Hobbies, Shopping): $1,000 (16.7%)
• Dining Out: $800 (13.3%)
• Savings & Debt Repayment: $1,200 (20.0%)

Financial Health Score: 6/10

Recommendations:
1. Reduce dining out from $800 to $500/month (8.3% of income) to free up $300 for emergency fund or retirement savings
2. Implement the 'cook 5, dine 2' rule: cook at home 5 days per week and allow dining out only 2 days to control restaurant spending
3. Set up automatic transfers of $1,200/month to savings on payday - aim for 3-6 months emergency fund ($18,000-$36,000) as first priority
4. Track all discretionary spending for 60 days using a budgeting app to identify additional areas where you can optimize spending
5. Consider meal prepping on weekends to reduce weeknight dining temptation and save an additional $200-300/month


In [32]:
%%writefile budget_agent.py
# Export complete budget agent implementation to Python file
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
from pydantic import BaseModel, Field
from typing import List
import matplotlib.pyplot as plt


# Define structured output models for financial data
class BudgetCategory(BaseModel):
    name: str = Field(description="Budget category name")
    amount: float = Field(description="Dollar amount for this category")
    percentage: float = Field(description="Percentage of total income")


class FinancialReport(BaseModel):
    monthly_income: float = Field(description="Total monthly income")
    budget_categories: List[BudgetCategory] = Field(
        description="List of budget categories"
    )
    recommendations: List[str] = Field(description="List of specific recommendations")
    financial_health_score: int = Field(
        ge=1, le=10, description="Financial health score from 1-10"
    )


# Enhanced system prompt for structured outputs
BUDGET_SYSTEM_PROMPT = """You are a helpful personal finance assistant. 
You provide general strategies for creating budgets, tips on financial discipline to achieve financial milestones, and analyze financial trends. You do not provide any investment advice. 

When generating financial reports, always provide:
1. Clear budget breakdowns using the 50/30/20 rule or custom allocations
2. Specific, actionable recommendations (2-3 steps)
3. A financial health score based on spending patterns
4. Practical budgeting and spending advice

Use structured output when requested to provide comprehensive financial reports."""

# Continue with previous configurations
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name="us-west-2",
    temperature=0.0,  # Deterministic responses for financial advice
)


@tool
def calculate_budget(monthly_income: float) -> str:
    """Calculate 50/30/20 budget breakdown for the given monthly income."""
    needs = monthly_income * 0.50
    wants = monthly_income * 0.30
    savings = monthly_income * 0.20
    return f"💰 Budget for ${monthly_income:,.0f}/month:\n• Needs: ${needs:,.0f} (50%)\n• Wants: ${wants:,.0f} (30%)\n• Savings: ${savings:,.0f} (20%)"


@tool
def create_financial_chart(
    data_dict: dict, chart_title: str = "Financial Chart"
) -> str:
    """Create a pie chart visualization from financial data dictionary."""
    if not data_dict:
        return "❌ No data provided for chart"

    labels = list(data_dict.keys())
    values = list(data_dict.values())
    colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", "#FF9FF3"]

    plt.figure(figsize=(8, 6))
    plt.pie(
        values,
        labels=labels,
        autopct="%1.1f%%",
        colors=colors[: len(values)],
        startangle=90,
    )
    plt.title(f"📊 {chart_title}", fontsize=14, fontweight="bold")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()

    return f"✅ {chart_title} visualization created!"


# Create our complete financial agent
budget_agent = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,
    tools=[calculate_budget, create_financial_chart, calculator],
    callback_handler=None,
)

if __name__ == "__main__":
    # Test structured output using new API
    print("\nStructured financial report:")
    structured_response = budget_agent(
        prompt="Generate a comprehensive financial report for someone earning $6000/month with $800 dining expenses.",
        structured_output_model=FinancialReport
    ).message.get("content")[0].get("toolUse").get("input")
    print(f"Income: ${structured_response['monthly_income']:,.0f}")
    for category in structured_response['budget_categories']:
        print(
            f"• {category['name']}: ${category['amount']:,.0f} ({category['percentage']:.1f}%)"
        )
    print(f"\nFinancial Health Score: {structured_response['financial_health_score']}/10")
    print("\nRecommendations:")
    for i, rec in enumerate(structured_response['recommendations'], 1):
        print(f"{i}. {rec}")

Writing budget_agent.py


In [33]:
# Test the exported budget agent implementation
!python budget_agent.py 


Structured financial report:
Traceback (most recent call last):
  File "/Users/alex/Developer/le-genai-ml/genai-agentcore-demos/finance-personal-assistant/workshop/budget_agent.py", line 99, in <module>
    ).message.get("content")[0].get("toolUse").get("input")
                                               ^^^
AttributeError: 'NoneType' object has no attribute 'get'
